# 04. Consumer Neo4j
Consumer buduje graf `USES` oraz `TRANSFER` z użyciem `MERGE` dla węzłów i `CREATE` dla zdarzeń przelewów, zgodnie z praktykami modelowania query-driven w Neo4j [file:4][file:5].

In [ ]:
from kafka import KafkaConsumer
from neo4j import GraphDatabase
import json

consumer = KafkaConsumer(
    "transactions",
    bootstrap_servers="kafka_streaming_lab:9092",
    auto_offset_reset="earliest",
    enable_auto_commit=True,
    group_id="transactions-neo4j-group",
    value_deserializer=lambda m: json.loads(m.decode("utf-8"))
)

driver = GraphDatabase.driver("bolt://neo4j_bank_lab:7687", auth=("neo4j", "test1234"))

setup_queries = [
    "CREATE CONSTRAINT user_name_unique IF NOT EXISTS FOR (u:User) REQUIRE u.name IS UNIQUE",
    "CREATE CONSTRAINT device_name_unique IF NOT EXISTS FOR (d:Device) REQUIRE d.name IS UNIQUE"
]

with driver.session() as session:
    for q in setup_queries:
        session.run(q)

cypher_query = """
MERGE (s:User {name: $sender})
MERGE (r:User {name: $receiver})
MERGE (ds:Device {name: $device_sender})
MERGE (dr:Device {name: $device_receiver})
MERGE (s)-[:USES]->(ds)
MERGE (r)-[:USES]->(dr)
CREATE (s)-[:TRANSFER {
    amount: $amount,
    timestamp: datetime($timestamp),
    title: $title
}]->(r)
"""

print("Kafka-to-Neo4j consumer running...")

with driver.session() as session:
    for msg in consumer:
        data = msg.value
        session.run(
            cypher_query,
            sender=data["sender"],
            receiver=data["receiver"],
            amount=float(data["amount"]),
            timestamp=data["timestamp"].replace(" ", ""),
            title=data["title"],
            device_sender=data["device_sender"],
            device_receiver=data["device_receiver"]
        )
        print(f"Inserted into Neo4j: {data}")